In [15]:
import pandas as pd
import numpy as np

# 1. Load all three CSV files
customers = pd.read_csv('customers.csv')
products = pd.read_csv('products.csv')
orders = pd.read_csv('orders.csv')

# 2. Display the dimensions of each DataFrame
print("--- 2. Dataset Dimensions ---")
print(f"Customers DataFrame shape: {customers.shape}")
print(f"Products DataFrame shape: {products.shape}")
print(f"Orders DataFrame shape: {orders.shape}")

# 3. Identify the columns in each dataset
print("\n--- 3. Dataset Columns ---")
print("Customers columns:", list(customers.columns))
print("Products columns:", list(products.columns))
print("Orders columns:", list(orders.columns))

# 4. Identify the common columns between datasets
print("\n--- 4. Common Columns ---")
print("Common between Orders & Customers:", list(set(orders.columns) & set(customers.columns)))
print("Common between Orders & Products:", list(set(orders.columns) & set(products.columns)))
print("Common between Customers & Products:", list(set(customers.columns) & set(products.columns)))

--- 2. Dataset Dimensions ---
Customers DataFrame shape: (3000, 5)
Products DataFrame shape: (100, 4)
Orders DataFrame shape: (15000, 5)

--- 3. Dataset Columns ---
Customers columns: ['Customer_ID', 'Customer_Name', 'City', 'Age', 'Gender']
Products columns: ['Product_ID', 'Product_Name', 'Category', 'Price']
Orders columns: ['Order_ID', 'Customer_ID', 'Product_ID', 'Quantity', 'Order_Date']

--- 4. Common Columns ---
Common between Orders & Customers: ['Customer_ID']
Common between Orders & Products: ['Product_ID']
Common between Customers & Products: []


In [16]:
# 6. Check each DataFrame for missing values
print("--- 6. Missing Values ---")
print("Customers missing values:\n", customers.isnull().sum())
print("\nProducts missing values:\n", products.isnull().sum())
print("\nOrders missing values:\n", orders.isnull().sum())

# 7. Check for duplicate records
print("\n--- 7. Duplicate Records ---")
print("Customers duplicates:", customers.duplicated().sum())
print("Products duplicates:", products.duplicated().sum())
print("Orders duplicates:", orders.duplicated().sum())

# 8. Check the validity of Customer_ID and Product_ID
print("\n--- 8. ID Validity ---")
print(f"Unique Customer_IDs in customers.csv: {customers['Customer_ID'].nunique()} (Total rows: {len(customers)})")
print(f"Unique Product_IDs in products.csv: {products['Product_ID'].nunique()} (Total rows: {len(products)})")

# 9. Convert Order_Date into datetime
orders['Order_Date'] = pd.to_datetime(orders['Order_Date'])

# 10. Check whether every order corresponds to a valid customer
invalid_customers = orders[~orders['Customer_ID'].isin(customers['Customer_ID'])]
print(f"\n--- 10. Referential Integrity: Customers ---")
print(f"Orders with invalid/missing Customer_IDs: {len(invalid_customers)}")

# 11. Check whether every order corresponds to a valid product
invalid_products = orders[~orders['Product_ID'].isin(products['Product_ID'])]
print(f"\n--- 11. Referential Integrity: Products ---")
print(f"Orders with invalid/missing Product_IDs: {len(invalid_products)}")

--- 6. Missing Values ---
Customers missing values:
 Customer_ID      0
Customer_Name    0
City             0
Age              0
Gender           0
dtype: int64

Products missing values:
 Product_ID      0
Product_Name    0
Category        0
Price           0
dtype: int64

Orders missing values:
 Order_ID       0
Customer_ID    0
Product_ID     0
Quantity       0
Order_Date     0
dtype: int64

--- 7. Duplicate Records ---
Customers duplicates: 0
Products duplicates: 0
Orders duplicates: 0

--- 8. ID Validity ---
Unique Customer_IDs in customers.csv: 3000 (Total rows: 3000)
Unique Product_IDs in products.csv: 100 (Total rows: 100)

--- 10. Referential Integrity: Customers ---
Orders with invalid/missing Customer_IDs: 0

--- 11. Referential Integrity: Products ---
Orders with invalid/missing Product_IDs: 0


In [17]:
# 12. Merge orders with customers
orders_customers = orders.merge(customers, on='Customer_ID', how='inner')

# 13. Merge the resulting DataFrame with products
df_final = orders_customers.merge(products, on='Product_ID', how='inner')

# 14. Display final consolidated DataFrame info and preview
print(f"Final Consolidated DataFrame Shape: {df_final.shape}")
print("\nFinal Columns:", list(df_final.columns))
display(df_final.head(3))

Final Consolidated DataFrame Shape: (15000, 12)

Final Columns: ['Order_ID', 'Customer_ID', 'Product_ID', 'Quantity', 'Order_Date', 'Customer_Name', 'City', 'Age', 'Gender', 'Product_Name', 'Category', 'Price']


,Order_ID,Customer_ID,Product_ID,Quantity,Order_Date,Customer_Name,City,Age,Gender,Product_Name,Category,Price
0,O000001,C00861,P0027,4,2025-01-01 00:00:00,Customer_861,Kolkata,51,Female,Product_27,Sports,56171.59
1,O000002,C01295,P0047,3,2025-01-01 02:00:00,Customer_1295,Bangalore,43,Male,Product_47,Clothing,80831.23
2,O000003,C01131,P0099,4,2025-01-01 04:00:00,Customer_1131,Pune,47,Female,Product_99,Electronics,83546.72


In [18]:
# 15. Create Order_Value
df_final['Order_Value'] = df_final['Price'] * df_final['Quantity']

# 16. Calculate total revenue
total_revenue = df_final['Order_Value'].sum()
print(f"16. Total Revenue: ${total_revenue:,.2f}")

# 17. Calculate average order value
avg_order_value = df_final['Order_Value'].mean()
print(f"17. Average Order Value (AOV): ${avg_order_value:,.2f}")

# 18. Find total revenue by product category
rev_by_category = df_final.groupby('Category')['Order_Value'].sum().sort_values(ascending=False)
print("\n18. Total Revenue by Product Category:\n", rev_by_category)

# 19. Find total revenue by city
rev_by_city = df_final.groupby('City')['Order_Value'].sum().sort_values(ascending=False)
print("\n19. Total Revenue by City:\n", rev_by_city)

# 20. Find the top 10 customers by spending
top_10_customers = df_final.groupby(['Customer_ID', 'Customer_Name'])['Order_Value'].sum().reset_index().sort_values(by='Order_Value', ascending=False).head(10)
print("\n20. Top 10 Customers by Spending:\n", top_10_customers)

# 21. Find the top 10 products by revenue
top_10_products = df_final.groupby(['Product_ID', 'Product_Name'])['Order_Value'].sum().reset_index().sort_values(by='Order_Value', ascending=False).head(10)
print("\n21. Top 10 Products by Revenue:\n", top_10_products)

# 22. Find the most frequently purchased product (by quantity)
most_freq_product = df_final.groupby(['Product_ID', 'Product_Name'])['Quantity'].sum().reset_index().sort_values(by='Quantity', ascending=False).head(1)
print("\n22. Most Frequently Purchased Product:\n", most_freq_product)

# 23. Find the city with the highest number of customers
city_highest_customers = customers.groupby('City')['Customer_ID'].nunique().sort_values(ascending=False).head(1)
print("\n23. City with Highest Number of Customers:\n", city_highest_customers)

# 24. Find the city generating the highest revenue
city_highest_revenue = rev_by_city.head(1)
print("\n24. City Generating Highest Revenue:\n", city_highest_revenue)

16. Total Revenue: $2,233,171,871.30
17. Average Order Value (AOV): $148,878.12

18. Total Revenue by Product Category:
 Category
Sports             5.426713e+08
Clothing           4.261489e+08
Home Appliances    3.313110e+08
Electronics        3.312926e+08
Books              3.050563e+08
Beauty             2.966918e+08
Name: Order_Value, dtype: float64

19. Total Revenue by City:
 City
Kolkata       3.155500e+08
Hyderabad     2.932639e+08
Coimbatore    2.831225e+08
Mumbai        2.803566e+08
Pune          2.797505e+08
Delhi         2.654541e+08
Bangalore     2.637059e+08
Chennai       2.519685e+08
Name: Order_Value, dtype: float64

20. Top 10 Customers by Spending:
      Customer_ID  Customer_Name  Order_Value
16        C00019    Customer_19   3176078.87
2872      C02897  Customer_2897   2528868.53
1896      C01915  Customer_1915   2476932.74
2688      C02709  Customer_2709   2400905.35
183       C00187   Customer_187   2394078.09
678       C00685   Customer_685   2352464.63
1283     

In [19]:
# 25. Divide customers into age groups
bins = [17, 30, 45, 60, np.inf]
labels = ['18–30', '31–45', '46–60', '61+']
df_final['Age_Group'] = pd.cut(df_final['Age'], bins=bins, labels=labels)

# 26. Calculate revenue generated by each age group
rev_by_age = df_final.groupby('Age_Group', observed=False)['Order_Value'].sum()
print("26. Revenue Generated by Each Age Group:\n", rev_by_age)

# 27. Compare spending between male and female customers
rev_by_gender = df_final.groupby('Gender')['Order_Value'].sum()
avg_spending_gender = df_final.groupby(['Customer_ID', 'Gender'])['Order_Value'].sum().groupby('Gender').mean()
print("\n27. Total Revenue by Gender:\n", rev_by_gender)
print("\nAverage Customer Spending by Gender:\n", avg_spending_gender)

# 28. Identify high-value customers (Top 10% / 90th percentile threshold)
cust_spending = df_final.groupby(['Customer_ID', 'Customer_Name'])['Order_Value'].sum().reset_index()
threshold_90 = cust_spending['Order_Value'].quantile(0.90)
high_value_customers = cust_spending[cust_spending['Order_Value'] >= threshold_90]
print(f"\n28. High-Value Customer Threshold (90th Percentile): ${threshold_90:,.2f}")
print(f"Total High-Value Customers: {len(high_value_customers)}")

# 29. Determine the percentage of customers who have placed more than one order
order_counts = df_final.groupby('Customer_ID')['Order_ID'].nunique()
repeat_customers_count = (order_counts > 1).sum()
total_active_customers = len(order_counts)
pct_repeat = (repeat_customers_count / total_active_customers) * 100
print(f"\n29. Percentage of Repeat Customers: {pct_repeat:.2f}% ({repeat_customers_count}/{total_active_customers})")

26. Revenue Generated by Each Age Group:
 Age_Group
18–30    5.586278e+08
31–45    6.223524e+08
46–60    6.599237e+08
61+      3.922680e+08
Name: Order_Value, dtype: float64

27. Total Revenue by Gender:
 Gender
Female    1.078972e+09
Male      1.154200e+09
Name: Order_Value, dtype: float64

Average Customer Spending by Gender:
 Gender
Female    738010.727907
Male      762351.510634
Name: Order_Value, dtype: float64

28. High-Value Customer Threshold (90th Percentile): $1,339,423.81
Total High-Value Customers: 298

29. Percentage of Repeat Customers: 96.81% (2881/2976)


In [20]:
# 30. Create a customer spending ranking
cust_spending['Spending_Rank'] = cust_spending['Order_Value'].rank(ascending=False, method='min').astype(int)
cust_spending = cust_spending.sort_values(by='Spending_Rank')
print("30. Customer Spending Ranking (Top 5):\n", cust_spending.head())

# 31. Identify the top 10% of customers based on total spending
top_10_pct_count = int(np.ceil(0.10 * len(cust_spending)))
top_10_pct_customers = cust_spending.head(top_10_pct_count)
print(f"\n31. Top 10% Customers Count: {top_10_pct_count}")

# 32. Find the most profitable product category for each city
city_cat_rev = df_final.groupby(['City', 'Category'])['Order_Value'].sum().reset_index()
most_prof_cat_city = city_cat_rev.loc[city_cat_rev.groupby('City')['Order_Value'].idxmax()]
print("\n32. Most Profitable Product Category for Each City:\n", most_prof_cat_city)

# 33. Find the most popular product in each category (by total quantity ordered)
cat_prod_qty = df_final.groupby(['Category', 'Product_ID', 'Product_Name'])['Quantity'].sum().reset_index()
most_pop_prod_cat = cat_prod_qty.loc[cat_prod_qty.groupby('Category')['Quantity'].idxmax()]
print("\n33. Most Popular Product in Each Category (by Quantity):\n", most_pop_prod_cat)

# 34. Calculate monthly revenue
df_final['Year_Month'] = df_final['Order_Date'].dt.to_period('M')
monthly_revenue = df_final.groupby('Year_Month')['Order_Value'].sum().reset_index()
print("\n34. Monthly Revenue (Preview):\n", monthly_revenue.head())

# 35. Identify the month with the highest revenue
highest_revenue_month = monthly_revenue.loc[monthly_revenue['Order_Value'].idxmax()]
print(f"\n35. Month with Highest Revenue: {highest_revenue_month['Year_Month']} (${highest_revenue_month['Order_Value']:,.2f})")

30. Customer Spending Ranking (Top 5):
      Customer_ID  Customer_Name  Order_Value  Spending_Rank
16        C00019    Customer_19   3176078.87              1
2872      C02897  Customer_2897   2528868.53              2
1896      C01915  Customer_1915   2476932.74              3
2688      C02709  Customer_2709   2400905.35              4
183       C00187   Customer_187   2394078.09              5

31. Top 10% Customers Count: 298

32. Most Profitable Product Category for Each City:
           City Category  Order_Value
5    Bangalore   Sports  71326795.18
11     Chennai   Sports  68298118.24
17  Coimbatore   Sports  69553033.93
23       Delhi   Sports  61019537.25
29   Hyderabad   Sports  65716665.57
35     Kolkata   Sports  71822933.47
41      Mumbai   Sports  63898188.25
47        Pune   Sports  71035989.75

33. Most Popular Product in Each Category (by Quantity):
            Category Product_ID Product_Name  Quantity
3            Beauty      P0010   Product_10       511
27          